# الدرس 17 - إنشاء وكلاء ذكاء اصطناعي محليين باستخدام Foundry Local و Qwen

في هذه المذكرة ستبني **مساعد هندسي محلي** يعمل بالكامل على محطة عملك. لا يتم استخدام الاستنتاج السحابي في أي وقت. يمكن للمساعد:

1. **استدعاء الأدوات** عبر استدعاء وظائف Qwen من خلال Foundry Local.
2. **قائمة وقراءة الملفات** داخل دليل مشروع معزول.
3. **تحليل الشيفرة البرمجية** باستخدام مقاييس بسيطة.
4. **البحث في الوثائق** باستخدام RAG محلي (Chroma).
5. **استخدام خادم MCP محلي** (يتخطى بأناقة إذا لم يتم تكوين أي منها).

يبدو كود الوكيل مشابهًا تقريبًا لدروس السحابة — فقط نقطة النهاية الخاصة بالعميل تنتقل من السحابة إلى `localhost`.


## الإعداد

قبل تشغيل هذا الدفتر:

1. **تثبيت Microsoft Foundry Local** (راجع [التوثيق](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) لنظام التشغيل الخاص بك).
2. **قم بتنزيل وبدء نموذج Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. قم بتثبيت حزم بايثون أدناه.

كل شيء يعمل محليًا. جهاز كمبيوتر بذاكرة ~8 جيجابايت هو الحد الأدنى الواقعي.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. الاتصال بـ Foundry Local

يقوم `FoundryLocalManager` بتنزيل النموذج إذا لزم الأمر، وتشغيل الخدمة المحلية، ويعطينا **نقطة نهاية متوافقة مع OpenAI**. ثم نشير إلى SDK standard الخاص بـ OpenAI عليها. مفتاح API هو عنصر نائب محلي — لا يتم استخدام بيانات اعتماد سحابية.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## ٢. الأدوات المحلية (عمليات الملفات في بيئة معزولة)

ننشئ مشروعًا نموذجيًا صغيرًا على القرص، ثم نحدد الأدوات المرتبطة بجذر هذا المشروع. تحقق البيئة المعزولة مهم حتى محليًا: الأداة التي تقرأ مسارات عشوائية تعمل بصلاحيات المستخدم الخاص بك ويمكنها الوصول إلى أي شيء يمكنك الوصول إليه.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG محلي مع Chroma

نقوم بتضمين مجموعة صغيرة من مقتطفات الوثائق في مجموعة Chroma محلية. يعمل Chroma داخل العملية ويخزن المتجهات على القرص — لا خادم، لا سحابة. أداة `search_docs` تسترجع أكثر المقتطفات صلةً بالاستعلام.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## ٤. حلقة استدعاء الأدوات

الآن نقوم بتسجيل الأدوات مع النموذج باستخدام مخطط أدوات OpenAI ونشغّل حلقة استدعاء الأدوات القياسية — يطلب النموذج أداة، نقوم بتنفيذها محليًا، نُغذي النتيجة مرة أخرى، ونكرر حتى ينتج النموذج إجابة نهائية. ميزة استدعاء الوظائف الموثوقة في Qwen هي ما يجعل هذا يعمل على الجهاز.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP محلي (اختياري)

MCP هو وسيلة نقل، وليس خدمة سحابية — يمكن تشغيل خادم MCP كعملية محلية عبر `stdio`. تُظهر الخلية أدناه كيفية الاتصال بخادم MCP المحلي إذا كان لديك واحد مكون (مثل خادم نظام الملفات). يتم تخطيه بسلاسة عندما لا يتم تعيين `LOCAL_MCP_COMMAND`، لذلك لا يزال المفكرة تعمل من البداية إلى النهاية بدونه.

ملاحظة أمنية: يعمل خادم MCP المحلي بصلاحيات المستخدم الخاص بك. قصره على دليل المشروع وتحقق من مخرجاته قبل التصرف بناءً عليها.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## ملخص

لقد بنيت مساعدًا هندسيًا يعمل بالكامل على جهازك:

- قدّم **Foundry Local** نموذج **Qwen** خلف نقطة نهاية متوافقة مع OpenAI — بحيث يتطابق رمز الوكيل مع دروس السحابة.
- وفرت **الأدوات المعزولة** للوكيل وصولًا إلى الملفات وتحليل الشيفرة دون مغادرة دليل المشروع.
- قدّم **Chroma** **RAG محلي** عبر الوثائق.
- أظهر **Local MCP** كيفية إعادة استخدام نظام MCP في وضع عدم الاتصال.

لم يُستخدم أي استدلال سحابي في أي نقطة.

### التحدي

وسّع الوكيل المحلي لــ:

1. **العمل مع عدة خوادم MCP** — ربط خادم نظام ملفات وخادم git ودع الوكيل يختار بينهما.
2. **استخدام الذاكرة المحلية** — الاحتفاظ بتاريخ محادثة قصيرة على القرص بحيث يتذكر المساعد الأدوار السابقة عبر إعادة تشغيل دفتر الملاحظات.
3. **دعم التنسيق المحلي متعدد الوكلاء** — إضافة وكيل محلي ثاني (مثل مراجعة) وجعل الاثنين يتعاونان في مهمة.

في الدرس التالي، ستتعلم كيفية تأمين وكلاء الذكاء الاصطناعي المنتشرين.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
